In [23]:
import json

# Remplace ici par ta réponse JSON complète si elle est dans un fichier ou une variable
response = {
    "features": [
        {
            "properties": {
                "vitesse_moyenne_vl": 20,
                "cleabs": "TRONROUT0000002203013981"
            }
        },
        {
            "properties": {
                "vitesse_moyenne_vl": 0,
                "cleabs": "TRONROUT0000002203014001"
            }
        }
    ]
}

# Extraction des vitesses
vma_dict = {}
for feature in response.get("features", []):
    props = feature.get("properties", {})
    vma = props.get("vitesse_moyenne_vl")
    troncon_id = props.get("cleabs")
    vma_dict[troncon_id] = vma

# Affichage
for troncon, vma in vma_dict.items():
    print(f"Tronçon {troncon} : {vma} km/h")

Tronçon TRONROUT0000002203013981 : 20 km/h
Tronçon TRONROUT0000002203014001 : 0 km/h


In [27]:

import requests

def get_vma_from_coords(x: float, y: float, buffer_m: float = 10.0):
    """
    Interroge l'API IGN pour obtenir la vitesse moyenne autorisée (vitesse_moyenne_vl)
    autour d'un point (x, y) dans un rayon (buffer) en mètres.

    Args:
        x (float): Longitude (WGS84)
        y (float): Latitude (WGS84)
        buffer_m (float): Rayon de recherche en mètres (défaut: 10m)

    Returns:
        dict: Dictionnaire {cleabs: vitesse_moyenne_vl}
    """
    url = "https://apicarto.ign.fr/api/wfs-geoportail/search"
    
    # Décalage de 15 mètres en longitude (approximation, valable pour petites distances)
    delta_long = 15 / 111320  # 1 degré ≈ 111320 m à l'équateur
    # Limiter les coordonnées à 6 chiffres après la virgule
    x_rounded = round(x, 6)
    y_rounded = round(y, 6)
    x2_rounded = round(x + delta_long, 6)
    geom = {
        "type": "LineString",
        "coordinates": [
            [x_rounded, y_rounded, 35],
            [x2_rounded, y_rounded, 35]  # Décalage de 15m en longitude
        ]
    }

    params = {
        "source": "BDTOPO_V3:troncon_de_route",
        "geom": str(geom).replace("'", '"'),
        "buffer": buffer_m
    }

    try:
        resp = requests.get(url, params=params)
        resp.raise_for_status()
        data = resp.json()

        vma_dict = {}
        for feature in data.get("features", []):
            props = feature.get("properties", {})
            cleabs = props.get("cleabs")
            vma = props.get("vitesse_moyenne_vl")
            if cleabs and vma is not None:
                vma_dict[cleabs] = vma

        return resp.url, vma_dict

    except requests.RequestException as e:
        print("Erreur de requête:", e)
        return {}

In [29]:
#x = 4.71794209
#y = 47.7888976

x=2.345017
y=48.859466
vma_result = get_vma_from_coords(x, y, 100.0)
if not vma_result:
    print("Aucune donnée de vitesse trouvée.")  
print("Vitesses autour du point :", vma_result)

Erreur de requête: 500 Server Error: Internal Server Error for url: https://apicarto.ign.fr/api/wfs-geoportail/search?source=BDTOPO_V3%3Atroncon_de_route&geom=%7B%22type%22%3A+%22LineString%22%2C+%22coordinates%22%3A+%5B%5B2.345017%2C+48.859466%2C+35%5D%2C+%5B2.345152%2C+48.859466%2C+35%5D%5D%7D&buffer=100.0
Aucune donnée de vitesse trouvée.
Vitesses autour du point : {}


In [30]:
def make_linestring_geom(x: float, y: float, delta_m: float = 15.0, z: float = 35.0):
    """
    Génère un objet geom de type LineString pour l'API IGN.

    Args:
        x (float): Longitude (WGS84)
        y (float): Latitude (WGS84)
        delta_m (float): Décalage en mètres sur la longitude (défaut: 15m)
        z (float): Altitude (défaut: 35)

    Returns:
        dict: Dictionnaire geom prêt à être utilisé dans la requête API
    """
    delta_long = delta_m / 111320  # Conversion mètres -> degrés longitude
    x2 = round(x + delta_long, 6)
    x1 = round(x, 6)
    y1 = round(y, 6)
    import urllib.parse
    geom = {
        "type": "LineString",
        "coordinates": [
            [x1, y1],
            [x2, y1]
        ]
    }
    # Retourne l'objet geom (dictionnaire), pas la chaîne encodée
    geom_str = str(geom).replace("'", '"')  # JSON-like string
    geom_encoded = urllib.parse.quote(geom_str, safe='')
    
    return geom_encoded


In [11]:
import urllib.parse

def make_square_multipolygon(x: float, y: float, delta_m: float = 10.0):
    """
    Génère un MultiPolygon carré centré sur (x, y) pour l'API IGN.

    Args:
        x (float): Longitude (WGS84)
        y (float): Latitude (WGS84)
        delta_m (float): Demi-côté du carré en mètres (défaut: 10m)

    Returns:
        str: MultiPolygon encodé pour l'URL
    """
    delta_deg = delta_m / 111320  # Conversion mètres -> degrés
    x1 = round(x - delta_deg, 6)
    x2 = round(x + delta_deg, 6)
    y1 = round(y - delta_deg, 6)
    y2 = round(y + delta_deg, 6)
    coords = [
        [x1, y2],
        [x1, y1],
        [x2, y1],
        [x2, y2],
        [x1, y2]  # On ferme le polygone
    ]
    multipolygon = {
        "type": "MultiPolygon",
        "coordinates": [[[coords]]]
    }
    multipolygon_str = str(multipolygon).replace("'", '"')
    multipolygon_encoded = urllib.parse.quote(multipolygon_str, safe='')
    return multipolygon_encoded

# Exemple d'utilisation
x = 2.345017
y = 48.859466
poly_encoded = make_square_multipolygon(x, y, delta_m=5)  # carré de 10m de côté
print(poly_encoded)

%7B%22type%22%3A%20%22MultiPolygon%22%2C%20%22coordinates%22%3A%20%5B%5B%5B%5B%5B2.344972%2C%2048.859511%5D%2C%20%5B2.344972%2C%2048.859421%5D%2C%20%5B2.345062%2C%2048.859421%5D%2C%20%5B2.345062%2C%2048.859511%5D%2C%20%5B2.344972%2C%2048.859511%5D%5D%5D%5D%5D%7D


In [31]:
# Affiche le résultat de la fonction make_linestring_geom
geom = make_linestring_geom(x=2.345017, y=48.859466, delta_m=15.0, z=35.0)
print(geom)
     
    

%7B%22type%22%3A%20%22LineString%22%2C%20%22coordinates%22%3A%20%5B%5B2.345017%2C%2048.859466%5D%2C%20%5B2.345152%2C%2048.859466%5D%5D%7D


In [32]:
#recherche de la vitesse moyenne autorisée (vma) pour un tronçon de route
#en se basant sur une ligne de x metres de long à partir d'un point donné

geom = make_linestring_geom(x=2.345017, y=48.859466, delta_m=20.0, z=35.0)
# geom est déjà encodé, pas besoin de le recalculer ici
url = f"https://apicarto.ign.fr/api/wfs-geoportail/search?source=BDTOPO_V3:troncon_de_route&geom={geom}"
headers = {"accept": "*/*"}
response = requests.get(url, headers=headers)
#print ("URL de la requête :", response.url)
try:
    data = response.json()
    if data.get("type") == "error":
        print("Erreur de l'API :", data.get("message"))
    else:
        for feature in data.get("features", []):
            props = feature.get("properties", {})
            vma = props.get("vitesse_moyenne_vl")
            if vma is not None:
                print("vitesse_moyenne_vl:", vma)
        #print(data)
except Exception as e:
    print("Erreur lors du décodage JSON ou de la requête :", e)


vitesse_moyenne_vl: 20
vitesse_moyenne_vl: 0


In [33]:
#{"type":"MultiPolygon","coordinates":[[[[-0.288863182067871,48.963666607295977],[-0.299592018127441,48.959299208576141],[-0.296330451965332,48.955325952385039],[-0.282125473022461,48.950675995388366],[-0.279722213745117,48.967019382922331],[-0.288863182067871,48.963666607295977]]]]}
def make_square_multipolygon(x: float, y: float, delta_m: float = 10.0):
    """
    Génère un MultiPolygon carré centré sur (x, y) pour l'API IGN.

    Args:
        x (float): Longitude (WGS84)
        y (float): Latitude (WGS84)
        delta_m (float): Demi-côté du carré en mètres (défaut: 10m)

    Returns:
        str: MultiPolygon encodé pour l'URL
    """
    delta_deg = delta_m / 111320  # Conversion mètres -> degrés
    x1 = round(x - delta_deg, 9)
    x2 = round(x + delta_deg, 9)
    y1 = round(y - delta_deg, 9)
    y2 = round(y + delta_deg, 9)
    coords = [
        [x1, y2],
        [x1, y1],
        [x2, y1],
        [x2, y2],
        [x1, y2]  # On ferme le polygone
    ]
    multipolygon = {
        "type": "MultiPolygon",
        "coordinates": [[coords]]  # <-- Un seul niveau ici !
    }
    multipolygon_str = str(multipolygon).replace("'", '"')
    multipolygon_encoded = urllib.parse.quote(multipolygon_str, safe='')
    return multipolygon_encoded

In [46]:
#Rechecrhe de la vitesse moyenne autorisée (vma) autour d'un point donné
#on trace un carré de xm de coté autours du point
# 
# 
from operator import ge


geom = make_square_multipolygon(x=2.32347100, y=48.86638600, delta_m=3)

# geom est déjà encodé, pas besoin de le recalculer ici
url = f"https://apicarto.ign.fr/api/wfs-geoportail/search?source=BDTOPO_V3:troncon_de_route&geom={geom}"
headers = {"accept": "*/*"}
response = requests.get(url, headers=headers)
print ("URL de la requête :", response.url)
try:
    data = response.json()
    if data.get("type") == "error":
        print("Erreur de l'API :", data.get("message"))
    else:
        for feature in data.get("features", []):
            props = feature.get("properties", {})
            vma = props.get("vitesse_moyenne_vl")
            larrout= props.get("largeur_de_chaussee")
            nbv = props.get("nombre_de_voies")
            agg= props.get("urbain")
            insee = props.get("insee_commune_gauche")
            adr=props.get("nom_voie_ban_gauche")
            


            if vma is not None:
                print(
                    "vitesse_moyenne_vl vma:", vma,
                    "larrout:", larrout,
                    "nbv:", nbv,
                    "agg:", agg,
                    "insee:", insee,
                    "adr:", adr
                )
        #print(data)
except Exception as e:
    print("Erreur lors du décodage JSON ou de la requête :", e)


URL de la requête : https://apicarto.ign.fr/api/wfs-geoportail/search?source=BDTOPO_V3:troncon_de_route&geom=%7B%22type%22%3A%20%22MultiPolygon%22%2C%20%22coordinates%22%3A%20%5B%5B%5B%5B2.323444051%2C%2048.866412949%5D%2C%20%5B2.323444051%2C%2048.866359051%5D%2C%20%5B2.323497949%2C%2048.866359051%5D%2C%20%5B2.323497949%2C%2048.866412949%5D%2C%20%5B2.323444051%2C%2048.866412949%5D%5D%5D%5D%7D
vitesse_moyenne_vl vma: 0 larrout: 24 nbv: 8 agg: True insee: 75108 adr: Place de la Concorde
vitesse_moyenne_vl vma: 0 larrout: None nbv: None agg: True insee: 75101 adr: Place de la Concorde


In [41]:
# recup du poluygone autour d'un point
polygonetest= make_square_multipolygon(x=2.32347100, y=48.86638600, delta_m=5)
print(polygonetest)

%7B%22type%22%3A%20%22MultiPolygon%22%2C%20%22coordinates%22%3A%20%5B%5B%5B%5B2.323426084%2C%2048.866430916%5D%2C%20%5B2.323426084%2C%2048.866341084%5D%2C%20%5B2.323515916%2C%2048.866341084%5D%2C%20%5B2.323515916%2C%2048.866430916%5D%2C%20%5B2.323426084%2C%2048.866430916%5D%5D%5D%5D%7D


In [17]:
!pip install geopy
#calcul d'une distance entre deux point gps
from geopy.distance import geodesic
def calculate_distance(coord1, coord2):
    """
    Calcule la distance entre deux points GPS en utilisant la formule de Haversine.

    Args:
        coord1 (tuple): Premier point (latitude, longitude).
        coord2 (tuple): Deuxième point (latitude, longitude).

    Returns:
        float: Distance en mètres.
    """
    return geodesic(coord1, coord2).meters


In [18]:
calculate_distance((48.86638600, 2.32347100), (48.86638600, 2.32347100))  # Distance entre le même point
calculate_distance((48.86638600, 2.32347100), (48.86638600, 2.32347100 + 0.0002))  # Distance de 11 mètres environ      
calculate_distance((46.08672900, 5.11126600), (46.08266499 , 5.10527743))  # Distance x y et gps PR 


646.9918312245288